# Jan Sahayak: Government Welfare Scheme Assistant using GenAI (RAG + LLM)
**RTD Open Elective Project (30 Marks) - Societal Use Case**

**Problem:** Millions of eligible citizens (farmers, women, students, senior citizens, workers) never claim welfare benefits because scheme information is scattered across portals, written in complex language, and hard to match to one's own situation.

**Solution:** An AI assistant that
1. takes a simple citizen profile (age, occupation, income, gender, state) and **finds schemes the person is eligible for** (rule-based filter),
2. answers **free-text questions** about schemes using **Retrieval-Augmented Generation** with a Large Language Model (Flan-T5), grounded in official scheme text,
3. explains benefits, documents and how to apply in **plain, simple language**.

**Pipeline:** Scheme database → eligibility filter → embeddings + FAISS retrieval → Flan-T5 grounded answer → Gradio UI

> Scheme details are summarised for demonstration. Verify on the official portal (e.g. myscheme.gov.in) before applying.

## Step 1: Install and import

In [ ]:
!pip install -q transformers sentence-transformers faiss-cpu gradio pandas matplotlib

In [ ]:
import time, torch, faiss, numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
print("GPU:", torch.cuda.is_available())

## Step 2: Scheme database
Each scheme has structured eligibility fields (used by the filter) and descriptive text (used by RAG). `min_age`/`max_age` of 0/120 means no limit; `occupation` of `any` means open to all; `gender` of `any` means open to all; `max_income` is annual family income in Rs (None = no limit).

In [ ]:
SCHEMES = [
 dict(name="PM-KISAN", category="Farmers", occupation="farmer", gender="any", min_age=18, max_age=120, max_income=None,
      benefit="Income support of Rs 6000 per year in three equal instalments of Rs 2000 directly into the bank account of landholding farmer families.",
      documents="Aadhaar card, land ownership records, bank account details linked with Aadhaar.",
      apply="Register on the PM-KISAN portal or through the nearest Common Service Centre (CSC) or the local agriculture office."),
 dict(name="Pradhan Mantri Fasal Bima Yojana", category="Farmers", occupation="farmer", gender="any", min_age=18, max_age=120, max_income=None,
      benefit="Crop insurance that gives financial support to farmers when crops are damaged by drought, flood, hailstorm, pests or disease. Farmers pay a low premium of 2 percent for kharif crops and 1.5 percent for rabi crops.",
      documents="Aadhaar, bank passbook, land records, sowing certificate.",
      apply="Apply through your bank, a CSC, or the PMFBY portal before the crop season cut-off date."),
 dict(name="Kisan Credit Card", category="Farmers", occupation="farmer", gender="any", min_age=18, max_age=75, max_income=None,
      benefit="Short-term credit for farmers to buy seeds, fertiliser and equipment at low interest. Interest subvention gives cheaper loans up to Rs 3 lakh when repaid on time.",
      documents="Identity proof, address proof, land documents, passport size photo.",
      apply="Apply at any nationalised bank, cooperative bank or regional rural bank."),
 dict(name="Ayushman Bharat PM-JAY", category="Health", occupation="any", gender="any", min_age=0, max_age=120, max_income=250000,
      benefit="Free health insurance cover of Rs 5 lakh per family per year for hospital treatment at empanelled public and private hospitals. It is cashless and paperless.",
      documents="Aadhaar card, ration card, and the PM-JAY e-card.",
      apply="Check eligibility on the PM-JAY website or at a CSC, then get the Ayushman card from an empanelled hospital or CSC."),
 dict(name="Pradhan Mantri Matru Vandana Yojana", category="Women", occupation="any", gender="female", min_age=19, max_age=45, max_income=None,
      benefit="Cash incentive of Rs 5000 in instalments to pregnant and lactating women for the first living child to help with nutrition and rest during pregnancy.",
      documents="Aadhaar card, bank passbook, mother-child protection card, identity proof.",
      apply="Register at the nearest Anganwadi Centre or health facility."),
 dict(name="Sukanya Samriddhi Yojana", category="Women", occupation="any", gender="female", min_age=0, max_age=10, max_income=None,
      benefit="A small savings scheme for the girl child with high interest and tax benefits. An account can be opened for a girl below 10 years with a minimum deposit of Rs 250 per year.",
      documents="Birth certificate of the girl child, parent or guardian Aadhaar and address proof.",
      apply="Open the account at a post office or authorised bank."),
 dict(name="PM Ujjwala Yojana", category="Women", occupation="any", gender="female", min_age=18, max_age=120, max_income=100000,
      benefit="Free LPG gas connection to women from poor households so that they do not have to cook on firewood and suffer smoke related illness.",
      documents="Aadhaar card, BPL ration card, bank account details, passport size photo.",
      apply="Submit the KYC form at the nearest LPG distributor."),
 dict(name="National Scholarship Portal - Post Matric Scholarship", category="Students", occupation="student", gender="any", min_age=15, max_age=30, max_income=250000,
      benefit="Scholarship for students of SC, ST and OBC categories studying after class 10 to cover tuition and maintenance allowance.",
      documents="Aadhaar card, income certificate, caste certificate, previous marksheet, bank account details, admission proof.",
      apply="Apply online on the National Scholarship Portal (scholarships.gov.in) before the yearly deadline."),
 dict(name="PM Vidyalaxmi Education Loan", category="Students", occupation="student", gender="any", min_age=16, max_age=35, max_income=None,
      benefit="Collateral-free and guarantor-free education loan for meritorious students admitted to top-ranked higher education institutions, with interest subvention for low income families.",
      documents="Admission letter, fee structure, Aadhaar, income proof, academic records.",
      apply="Apply through the PM-Vidyalaxmi portal, which sends the application to participating banks."),
 dict(name="Atal Pension Yojana", category="Workers", occupation="worker", gender="any", min_age=18, max_age=40, max_income=None,
      benefit="Guaranteed monthly pension of Rs 1000 to Rs 5000 after the age of 60 depending on the contribution. It is meant for workers in the unorganised sector.",
      documents="Aadhaar, mobile number, savings bank account.",
      apply="Open an Atal Pension account in your bank or post office savings account."),
 dict(name="e-Shram Card", category="Workers", occupation="worker", gender="any", min_age=16, max_age=59, max_income=None,
      benefit="National database registration for unorganised workers such as construction workers, street vendors and domestic workers. It gives accident insurance cover of Rs 2 lakh.",
      documents="Aadhaar card, mobile number linked with Aadhaar, bank account.",
      apply="Register free on the e-Shram portal or at any CSC. Workers must not be income tax payers or EPFO/ESIC members."),
 dict(name="PM SVANidhi", category="Workers", occupation="vendor", gender="any", min_age=18, max_age=120, max_income=None,
      benefit="Working capital loan of Rs 10000 for street vendors, which can be repaid in one year, with interest subsidy on timely repayment and cashback for digital transactions.",
      documents="Certificate of vending or identity card, Aadhaar, bank account.",
      apply="Apply on the PM SVANidhi portal or through a bank, urban local body or CSC."),
 dict(name="Indira Gandhi National Old Age Pension", category="Senior citizens", occupation="any", gender="any", min_age=60, max_age=120, max_income=100000,
      benefit="Monthly pension to senior citizens belonging to below poverty line households. Central assistance is Rs 200 per month for age 60 to 79 and Rs 500 per month for age 80 and above, and many states add more.",
      documents="Age proof, BPL certificate, Aadhaar, bank passbook.",
      apply="Apply at the block development office, municipality office or the state social welfare portal."),
 dict(name="Pradhan Mantri Awas Yojana", category="Housing", occupation="any", gender="any", min_age=18, max_age=120, max_income=300000,
      benefit="Financial assistance to build or buy a pucca house for economically weaker families who do not own a house. Urban beneficiaries also get a home loan interest subsidy.",
      documents="Aadhaar, income certificate, proof that the family owns no pucca house, bank account details.",
      apply="Apply through the PMAY portal, the urban local body, the gram panchayat or a CSC."),
 dict(name="Mahatma Gandhi NREGA", category="Employment", occupation="any", gender="any", min_age=18, max_age=120, max_income=None,
      benefit="Guarantees 100 days of wage employment per year to rural households whose adult members are willing to do unskilled manual work.",
      documents="Job card, Aadhaar, bank or post office account, passport size photo.",
      apply="Apply for a job card at the gram panchayat, then submit a written work application."),
 dict(name="Skill India - PM Kaushal Vikas Yojana", category="Skill training", occupation="any", gender="any", min_age=15, max_age=45, max_income=None,
      benefit="Free short-term skill training with certification and a monetary reward on successful assessment so that youth can find better jobs.",
      documents="Aadhaar card, bank account details, education certificate.",
      apply="Enrol on the Skill India portal or at the nearest PMKVY training centre."),
]
df = pd.DataFrame(SCHEMES)
print(len(df), "schemes")
df[["name", "category"]]

## Step 3: Eligibility filter (rule-based)

In [ ]:
def eligible_schemes(age, gender, occupation, income, state=None):
    gender, occupation = gender.lower(), occupation.lower()
    out = []
    for s in SCHEMES:
        ok  = s["min_age"] <= age <= s["max_age"]
        ok &= s["gender"] in ("any", gender)
        ok &= s["occupation"] in ("any", occupation)
        ok &= (s["max_income"] is None) or (income <= s["max_income"])
        if ok: out.append(s)
    return out

# demo
for s in eligible_schemes(age=34, gender="female", occupation="farmer", income=90000):
    print("-", s["name"])

## Step 4: Embeddings + FAISS vector index for RAG

In [ ]:
docs = [f"{s['name']} ({s['category']}). Benefit: {s['benefit']} Documents needed: {s['documents']} How to apply: {s['apply']}" for s in SCHEMES]
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb = embedder.encode(docs, normalize_embeddings=True).astype("float32")
index = faiss.IndexFlatIP(emb.shape[1]); index.add(emb)
print("Indexed", index.ntotal, "schemes")

## Step 5: Load the LLM

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "google/flan-t5-base"      # flan-t5-large gives better answers on GPU
tok = AutoTokenizer.from_pretrained(MODEL)
llm = AutoModelForSeq2SeqLM.from_pretrained(MODEL).to(device)

def generate(prompt, max_new=120):
    inp = tok(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = llm.generate(**inp, max_new_tokens=max_new, num_beams=4, early_stopping=True)
    return tok.decode(out[0], skip_special_tokens=True)

## Step 6: RAG question answering + plain-language explanation

In [ ]:
THRESH = 0.30
def retrieve(q, k=2):
    v = embedder.encode([q], normalize_embeddings=True).astype("float32")
    sc, ids = index.search(v, k)
    return [(SCHEMES[i], float(s)) for i, s in zip(ids[0], sc[0])]

def ask(question):
    hits = retrieve(question)
    if hits[0][1] < THRESH:
        return "I could not find this in my scheme records. Please visit myscheme.gov.in or your nearest Common Service Centre (CSC).", hits
    ctx = " ".join(docs[SCHEMES.index(h[0])] for h in hits)
    p = ("Answer the question in simple words using only the context.\n\n"
         f"Context: {ctx}\n\nQuestion: {question}\nAnswer:")
    return generate(p), hits

def explain(s):
    """Plain-language summary of one scheme for a citizen with low literacy."""
    p = (f"Explain this government scheme in two simple sentences for a villager. "
         f"Scheme: {s['name']}. Benefit: {s['benefit']}")
    return generate(p, 80)

for q in ["How much money does a farmer get every year from the government?",
          "What documents do I need for e-Shram card?",
          "Which scheme gives free gas connection?"]:
    a, h = ask(q)
    print("Q:", q, "\nA:", a, "\nScheme:", h[0][0]["name"], f"({h[0][1]:.2f})\n")

## Step 7: Full citizen assistant (profile -> eligible schemes -> explanation)

In [ ]:
def assistant(age, gender, occupation, income):
    es = eligible_schemes(age, gender, occupation, income)
    if not es:
        return "No matching schemes found for this profile. Please try different details or visit myscheme.gov.in."
    out = [f"You may be eligible for {len(es)} scheme(s):\n"]
    for s in es:
        out.append(f"### {s['name']}  ({s['category']})\n{explain(s)}\n**Benefit:** {s['benefit']}\n**Documents:** {s['documents']}\n**How to apply:** {s['apply']}\n")
    return "\n".join(out)

print(assistant(65, "female", "any", 60000)[:1500])

## Step 8: Evaluation

In [ ]:
# (a) Eligibility engine: profile -> scheme that MUST appear
elig_tests = [
 (dict(age=40, gender="male", occupation="farmer", income=100000), "PM-KISAN"),
 (dict(age=28, gender="female", occupation="any", income=50000), "PM Ujjwala Yojana"),
 (dict(age=70, gender="male", occupation="any", income=50000), "Indira Gandhi National Old Age Pension"),
 (dict(age=20, gender="male", occupation="student", income=150000), "National Scholarship Portal - Post Matric Scholarship"),
 (dict(age=30, gender="male", occupation="worker", income=80000), "e-Shram Card"),
 (dict(age=35, gender="male", occupation="vendor", income=70000), "PM SVANidhi"),
 (dict(age=8, gender="female", occupation="any", income=200000), "Sukanya Samriddhi Yojana"),
]
elig_acc = np.mean([t[1] in [s["name"] for s in eligible_schemes(**t[0])] for t in elig_tests])

# (b) Retrieval: question -> expected scheme in top-2
ret_tests = [
 ("How much money do farmers get every year?", "PM-KISAN"),
 ("Free health insurance of 5 lakh", "Ayushman Bharat PM-JAY"),
 ("Cash help for pregnant women", "Pradhan Mantri Matru Vandana Yojana"),
 ("Savings account for my daughter", "Sukanya Samriddhi Yojana"),
 ("Loan for street vendors", "PM SVANidhi"),
 ("100 days of work in village", "Mahatma Gandhi NREGA"),
 ("Pension for unorganised workers after 60", "Atal Pension Yojana"),
 ("Crop damage insurance", "Pradhan Mantri Fasal Bima Yojana"),
 ("Help to build a house for poor family", "Pradhan Mantri Awas Yojana"),
 ("Free skill training with certificate", "Skill India - PM Kaushal Vikas Yojana"),
 ("Education loan without collateral", "PM Vidyalaxmi Education Loan"),
 ("Scholarship for SC ST OBC students", "National Scholarship Portal - Post Matric Scholarship"),
]
hits_ok, times = [], []
for q, exp in ret_tests:
    t = time.time(); a, h = ask(q); times.append(time.time() - t)
    hits_ok.append(exp in [x[0]["name"] for x in h])
ret_acc = np.mean(hits_ok)
print(f"Eligibility accuracy : {elig_acc*100:.0f}%")
print(f"Retrieval accuracy   : {ret_acc*100:.0f}%")
print(f"Avg response time    : {np.mean(times):.2f}s")

import matplotlib.pyplot as plt
plt.bar(["Eligibility engine", "RAG retrieval (top-2)"], [elig_acc*100, ret_acc*100], color=["#4C72B0", "#55A868"])
plt.ylim(0, 110); plt.ylabel("Accuracy (%)"); plt.title("Jan Sahayak evaluation"); plt.show()

## Step 9: Web app (Gradio)

In [ ]:
import gradio as gr

def chat_fn(message, history):
    a, h = ask(message)
    return a + f"\n\n(Source scheme: {h[0][0]['name']})"

with gr.Blocks(title="Jan Sahayak") as demo:
    gr.Markdown("# Jan Sahayak - Government Scheme Assistant")
    with gr.Tab("Find my schemes"):
        age = gr.Number(label="Age", value=35, precision=0)
        gender = gr.Dropdown(["male", "female"], label="Gender", value="female")
        occ = gr.Dropdown(["farmer", "student", "worker", "vendor", "any"], label="Occupation", value="farmer")
        inc = gr.Number(label="Annual family income (Rs)", value=100000)
        out = gr.Markdown()
        gr.Button("Find schemes").click(lambda a, g, o, i: assistant(int(a), g, o, i), [age, gender, occ, inc], out)
    with gr.Tab("Ask a question"):
        gr.ChatInterface(chat_fn, examples=["How do I apply for PM-KISAN?", "Which documents are needed for Ayushman card?"])
demo.launch(share=True)

## Societal Impact
- **Bridges the information gap** for farmers, women, students, senior citizens and informal workers.
- **Saves time and travel:** a citizen finds all eligible schemes in one place instead of visiting multiple offices.
- **Reduces middlemen and misinformation** by giving documents and official application steps.
- Aligned with UN SDGs: No Poverty (1), Good Health (3), Quality Education (4), Gender Equality (5), Reduced Inequalities (10).

## Limitations
- Sample database of 16 schemes; state-specific schemes and rule changes need regular updates.
- Flan-T5 may give short answers; a larger LLM improves fluency. The threshold prevents answers on unknown topics.
- Not a legal or official eligibility decision - always verify on the official portal.

## Future Scope
Hindi/Bengali/regional-language and voice interface (for low-literacy users), WhatsApp deployment, auto-scraping of myscheme.gov.in, document upload with OCR to pre-fill forms, state-wise scheme support.

**References:** Lewis et al. (2020) *RAG for Knowledge-Intensive NLP*; Chung et al. (2022) *Flan-T5*; Reimers & Gurevych (2019) *Sentence-BERT*; Johnson et al. (2019) *FAISS*; Government of India, myscheme.gov.in.